# 05 — Figures 3 and 4: number of flares per day

* **Figure 3** — total number of flares per day, AR 13663 vs AR 13664 (line plot).
* **Figure 4 (a, b)** — number of flares per day broken down by class (C, M, X) for each AR (grouped bar charts).

**Inputs:**
* Aggregated counts hard-coded below (consistent with the curated tables in `data/dados_analises_may_2024_flares.xlsx`, sheet `qtd_flare_per_day`).
* Per-class breakdown from the same workbook (sheet `Class_flare_per_day`).

**Outputs:**
* `figures/amount_flare_over_days_revised.png` (Fig. 3)
* `figures/number_flares_class_AR13663.png` (Fig. 4a)
* `figures/number_flares_class_AR13664.png` (Fig. 4b)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path('..').resolve() / 'data'
FIG_DIR  = Path('..').resolve() / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

XLSX = DATA_DIR / 'dados_analises_may_2024_flares_pt-br_novo.xlsx'


## Figure 3 — total flares per day


In [ ]:
dates = ['2024-05-01', '2024-05-02', '2024-05-03', '2024-05-04',
         '2024-05-05', '2024-05-06', '2024-05-07', '2024-05-08',
         '2024-05-09', '2024-05-10', '2024-05-11', '2024-05-12',
         '2024-05-13', '2024-05-14', '2024-05-15']

df = pd.DataFrame({
    'Date':     pd.to_datetime(dates),
    'AR 13663': [4, 5, 8, 18, 15, 16, 7, 5, 0, 1, 0, 0, 0, 0, 0],
    'AR 13664': [0, 6, 4,  1,  5,  0, 2, 9, 9, 13, 8, 7, 13, 4, 1],
})
df['Total'] = df['AR 13663'] + df['AR 13664']
df


In [ ]:
plt.figure(figsize=(12, 7))
plt.plot(df['Date'], df['AR 13663'], label='AR 13663', color='orange')
plt.plot(df['Date'], df['AR 13664'], label='AR 13664', color='purple')

plt.title('Number of Flares over the days', color='blue', fontsize=16)
plt.ylabel('Number of flares', fontsize=14)
plt.ylim(0, 20)
plt.yticks(range(0, 20, 2), fontsize=16)
plt.xticks(rotation=45, fontsize=16)
plt.legend(fontsize=14, loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig(FIG_DIR / 'amount_flare_over_days_revised.png',
            dpi=300, bbox_inches='tight')
plt.show()


## Figure 4 — per-class breakdown

Reads sheet `Class_flare_per_day` from the curated workbook. Each AR occupies two consecutive blocks of columns:
* AR 13663 → `'AR 13663', 'Classe C', 'Classe M', 'Classe X'`
* AR 13664 → `'AR 13664', 'Classe C.1', 'Classe M.1', 'Classe X.1'`


In [ ]:
data = pd.read_excel(XLSX, sheet_name='Class_flare_per_day')
data.head()


In [ ]:
def plot_per_class(df_block, ar_label, end_date, outfile):
    df_block = df_block.copy()
    df_block['Date'] = pd.to_datetime(df_block['Date'])
    df_block = df_block[df_block['Date'] <= end_date]

    x = np.arange(len(df_block['Date']))
    w = 0.25
    plt.figure(figsize=(12, 6))
    plt.bar(x - w, df_block['Classe C'], w, label='C', color='blue',  edgecolor='black')
    plt.bar(x,     df_block['Classe M'], w, label='M', color='green', edgecolor='black')
    plt.bar(x + w, df_block['Classe X'], w, label='X', color='red',   edgecolor='black')

    plt.ylabel('Number of Flares', fontsize=16)
    plt.title(f'Number of Flares per Day by Class ({ar_label})', fontsize=16)
    plt.legend(title='Flare Class', fontsize=14)
    plt.xticks(x, df_block['Date'].dt.strftime('%Y-%m-%d'), rotation=45, fontsize=14)
    plt.tight_layout()
    plt.savefig(outfile, dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# AR 13663 — up to May 10 (after that the region was off-limb)
block = data[['AR 13663', 'Classe C', 'Classe M', 'Classe X']].copy()
block.rename(columns={'AR 13663': 'Date'}, inplace=True)
plot_per_class(block, 'AR 13663', '2024-05-10',
               FIG_DIR / 'number_flares_class_AR13663.png')


In [ ]:
# AR 13664 — full window May 01 – 15
block = data[['AR 13664', 'Classe C.1', 'Classe M.1', 'Classe X.1']].copy()
block.rename(columns={
    'AR 13664':  'Date',
    'Classe C.1': 'Classe C',
    'Classe M.1': 'Classe M',
    'Classe X.1': 'Classe X',
}, inplace=True)
plot_per_class(block, 'AR 13664', '2024-05-15',
               FIG_DIR / 'number_flares_class_AR13664.png')
